# CaT-GNN Fraud Detection — IEEE-CIS (GTAN Graph + Causal Temporal GNN)

**Architecture basis:** `architecture_justification_v2.md` §4.5  
**Preprocessing:** identical to `experiment_tcn.ipynb`  
**Graph construction:** GTAN methodology, identical to `experiment_gat.ipynb`  
**Model:** CausalInspector (GATConv attention scoring + split-aggregate) +  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;CausalIntervener (PL variant — learned blend, training-only intervention)  

**Reference:** Duan et al., *CaT-GNN: Enhancing Credit Card Fraud Detection via Causal Temporal Graph Neural Networks* (arXiv:2402.14708, Nov 2024)


In [ ]:
# Install DGL — uncomment the correct line for your environment
# Kaggle (PyTorch 2.x + CUDA 11.8):
# !pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/cu118/repo.html -q
# CPU-only fallback:
# !pip install dgl -f https://data.dgl.ai/wheels/repo.html -q


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import GATConv
import dgl.dataloading

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)

import warnings
warnings.filterwarnings('ignore')
import os, json, gc, time, datetime
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'DGL version: {dgl.__version__}')
print(f'PyTorch version: {torch.__version__}')


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# Graph construction (GTAN methodology — arch_justification_v2 §4.2)
GRAPH_K_NEIGHBORS = 3
GRAPH_RELATION_COLS = ['uid', 'card1', 'P_emaildomain', 'DeviceInfo']
MAX_GROUP_SIZE = 5000

# CaT-GNN model (arch_justification_v2 §4.5)
HIDDEN_DIM = 64
NUM_HEADS  = 4
FEAT_DROP  = 0.3
ATTN_DROP  = 0.3
ENV_RATIO  = 0.5

# Training hyperparameters
# LR raised from 0.0005 → 0.001: paper uses 0.003; 0.0005 caused underfitting
# FAN_OUT raised [10] → [15]: more neighbors = more stable causal/env split
BATCH_SIZE   = 512
LR           = 0.001          # raised from 0.0005 (too slow convergence)
EPOCHS       = 200
PATIENCE_ES  = 50
PATIENCE_LR  = 10
LR_FACTOR    = 0.5
MIN_LR       = 1e-7
FAN_OUT      = [15]           # raised from [10]: more neighbors for split

SAVE_DIR = 'catgnn_results'
import os
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'models'), exist_ok=True)

print('Configuration:')
print(f'  [ARCH v2]  Graph K={GRAPH_K_NEIGHBORS}, Relations: {GRAPH_RELATION_COLS}')
print(f'  [ARCH v2]  CaT-GNN: hidden={HIDDEN_DIM}, heads={NUM_HEADS}, env_ratio={ENV_RATIO}')
print(f'  [IMPROVED] LR={LR} (was 0.0005), FAN_OUT={FAN_OUT} (was [10])')
print(f'  [TCN REF]  EarlyStopping patience={PATIENCE_ES}, ReduceLROnPlateau patience={PATIENCE_LR}')


---
## 1. Data Loading & Preprocessing

**Identical** to `experiment_tcn.ipynb` and `experiment_gat.ipynb` — same column selection, V_COLS, D_SKIP={1,2,3,5,9}, DT_M calendar formula, UID construction, and all encode_FE/AG/AG2 calls. Only difference: we don't reshape to sequences (TCN) or rename to X_gat — we use X_catgnn.

We retain `uid`, `card1`, `P_emaildomain`, `DeviceInfo` for graph construction before dropping them from node features.


In [ ]:
# Load raw data
X_train  = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
X_test   = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_transaction.csv')
train_id = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')
test_id  = pd.read_csv('/kaggle/input/ieee-fraud-detection/test_identity.csv')

# Normalize identity column names: id-01 (dash) vs id_01 (underscore)
for df in [train_id, test_id, X_train, X_test]:
    df.columns = [c.replace('-', '_') if c.startswith('id') else c
                  for c in df.columns]

X_train = X_train.merge(train_id, on='TransactionID', how='left')
X_test  = X_test.merge(test_id,  on='TransactionID', how='left')

y_train = X_train['isFraud'].copy()
del X_train['isFraud']

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f'Train: {X_train.shape},  Test: {X_test.shape}')
print(f'Fraud rate: {y_train.mean()*100:.2f}%')


In [ ]:
# Column definitions (same as experiment_tcn) ─────────────────────────────
V_COLS  = [1,3,4,6,8,11]
V_COLS += [13,14,17,20,23,26,27,30]
V_COLS += [36,37,40,41,44,47,48]
V_COLS += [54,56,59,62,65,67,68,70]
V_COLS += [76,78,80,82,86,88,89,91]
V_COLS += [107,108,111,115,117,120,121,123]
V_COLS += [124,127,129,130,136]
V_COLS += [138,139,142,147,156,162]
V_COLS += [165,160,166]
V_COLS += [178,176,173,182]
V_COLS += [187,203,205,207,215]
V_COLS += [169,171,175,180,185,188,198,210,209]
V_COLS += [218,223,224,226,228,229,235]
V_COLS += [220,221,234,238,250,271]
V_COLS += [240,258,257,253,252,260,261]
V_COLS += [264,266,267,274,277]
V_COLS += [294,284,285,286,291,297]
V_COLS += [303,305,307,309,310,320]
V_COLS += [281,283,289,296,301,314]

all_cols = list(X_train.columns)
v_to_drop = [c for c in all_cols if c.startswith('V')
             and c not in [f'V{i}' for i in V_COLS]]
X_train.drop(columns=v_to_drop, inplace=True)
X_test.drop(columns=v_to_drop, inplace=True)

CAT_COLS = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'id_12','id_15','id_16','id_23','id_27','id_28','id_29',
    'id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38',
    'DeviceType','DeviceInfo'
]

print(f'V_COLS count: {len(set(V_COLS))}')
print(f'Shape after V-col drop: {X_train.shape}')


In [ ]:
# Sort by TransactionDT — attach label first to keep alignment
X_train['__label__'] = y_train.values
X_train = X_train.sort_values('TransactionDT').reset_index(drop=True)
y_train = X_train.pop('__label__').astype('int8')
X_test  = X_test.sort_values('TransactionDT').reset_index(drop=True)

# day column (used for UID and D-transform)
X_train['day'] = X_train['TransactionDT'] / np.float32(24 * 60 * 60)
X_test['day']  = X_test['TransactionDT']  / np.float32(24 * 60 * 60)

# D-column transform — experiment_tcn exact approach:
#   formula : D_new = D_old - day
#   SKIP D1, D2, D3, D5, D9 — these keep their original values.
#   D1 must NOT be transformed: UID uses floor(day - D1_orig).
D_SKIP = {1, 2, 3, 5, 9}
for i in range(1, 16):
    col = f'D{i}'
    if i in D_SKIP:
        continue
    if col in X_train.columns:
        X_train[col] = X_train[col] - X_train['day']
    if col in X_test.columns:
        X_test[col]  = X_test[col]  - X_test['day']

# DT_M — experiment_tcn exact formula (calendar month, not 30-day approximation)
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
for df in [X_train, X_test]:
    dt_series = df['TransactionDT'].apply(
        lambda x: START_DATE + datetime.timedelta(seconds=x))
    df['DT_M'] = (dt_series.dt.year - 2017) * 12 + dt_series.dt.month

# Factorize categoricals + shift numerics positive, NaN -> -1
SKIP = {'TransactionAmt', 'TransactionDT', 'day', 'DT_M'}
shared_cols = set(X_train.columns) & set(X_test.columns)

for f in list(X_train.columns):
    if (str(X_train[f].dtype) == 'category') or (X_train[f].dtype == 'object'):
        if f in shared_cols:
            df_comb = pd.concat([X_train[f], X_test[f]], axis=0)
            df_comb, _ = df_comb.factorize(sort=True)
            if df_comb.max() > 32000:
                X_train[f] = df_comb[:len(X_train)].astype('int32')
                X_test[f]  = df_comb[len(X_train):].astype('int32')
            else:
                X_train[f] = df_comb[:len(X_train)].astype('int16')
                X_test[f]  = df_comb[len(X_train):].astype('int16')
        else:
            X_train[f], _ = X_train[f].factorize(sort=True)
            X_train[f] = X_train[f].astype('int16')
    elif f not in SKIP:
        if f in shared_cols:
            mn = np.min((X_train[f].min(), X_test[f].min()))
            X_train[f] -= np.float32(mn)
            X_test[f]  -= np.float32(mn)
            X_train[f].fillna(-1, inplace=True)
            X_test[f].fillna(-1, inplace=True)
        else:
            mn = X_train[f].min()
            X_train[f] -= np.float32(mn)
            X_train[f].fillna(-1, inplace=True)

print(f'Sorted + preprocessed. X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'Fraud rate after sort: {y_train.mean()*100:.2f}%')
print(f'DT_M range: {X_train["DT_M"].min()} to {X_train["DT_M"].max()}')


In [ ]:
# ── Encoding helper functions (same as experiment_tcn) ────────────────────
def encode_LE(col, verbose=True, df1=None, df2=None):
    if df1 is None: df1 = X_train
    if df2 is None: df2 = X_test
    df_comb = pd.concat([df1[col], df2[col]], axis=0)
    df_comb, _ = df_comb.factorize(sort=True)
    if df_comb.max() > 32000:
        df1[col] = df_comb[:len(df1)].astype('int32')
        df2[col] = df_comb[len(df1):].astype('int32')
    else:
        df1[col] = df_comb[:len(df1)].astype('int16')
        df2[col] = df_comb[len(df1):].astype('int16')
    if verbose: print(col, ', ', end='')

def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col], df2[col]])
        vc = df.value_counts(dropna=True, normalize=True).to_dict()
        vc[-1] = -1
        nm = col + '_FE'
        df1[nm] = df1[col].map(vc).astype('float32')
        df2[nm] = df2[col].map(vc).astype('float32')
        print(nm, ', ', end='')

def encode_AG(main_columns, uids, aggregations=['mean'], train_df=None, test_df=None,
              fillna=True, usena=False):
    if train_df is None: train_df = X_train
    if test_df  is None: test_df  = X_test
    for main_column in main_columns:
        for col in uids:
            for agg_type in aggregations:
                new_col_name = main_column + '_' + col + '_' + agg_type
                temp_df = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]])
                if usena:
                    temp_df.loc[temp_df[main_column] == -1, main_column] = np.nan
                temp_df = temp_df.groupby([col])[main_column].agg([agg_type]).reset_index()
                temp_df = temp_df.rename(columns={agg_type: new_col_name})
                temp_df.index = list(temp_df[col])
                temp_df = temp_df[new_col_name].to_dict()
                train_df[new_col_name] = train_df[col].map(temp_df).astype('float32')
                test_df[new_col_name]  = test_df[col].map(temp_df).astype('float32')
                if fillna:
                    train_df[new_col_name].fillna(-1, inplace=True)
                    test_df[new_col_name].fillna(-1, inplace=True)
                print("'" + new_col_name + "'", ', ', end='')

def encode_AG2(main_columns, uids, train_df=None, test_df=None):
    if train_df is None: train_df = X_train
    if test_df  is None: test_df  = X_test
    for main_column in main_columns:
        for col in uids:
            comb = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]], axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            train_df[col + '_' + main_column + '_ct'] = train_df[col].map(mp).astype('float32')
            test_df[col + '_' + main_column + '_ct']  = test_df[col].map(mp).astype('float32')
            print(col + '_' + main_column + '_ct, ', end='')

def encode_CB(col1, col2, df1=None, df2=None):
    if df1 is None: df1 = X_train
    if df2 is None: df2 = X_test
    nm = col1 + '_' + col2
    df1[nm] = df1[col1].astype(str) + '_' + df1[col2].astype(str)
    df2[nm] = df2[col1].astype(str) + '_' + df2[col2].astype(str)
    encode_LE(nm, verbose=False)

print('Encoding functions defined.')


In [ ]:
# ── Feature engineering (same as experiment_tcn) ─────────────────────────
X_train['cents'] = (X_train['TransactionAmt'] - np.floor(X_train['TransactionAmt'])).astype('float32')
X_test['cents']  = (X_test['TransactionAmt']  - np.floor(X_test['TransactionAmt'])).astype('float32')

encode_CB('card1', 'addr1')
encode_CB('card1_addr1', 'P_emaildomain')

# UID construction: card1_addr1 + floor(day - D1)
# D1 was NOT transformed (D_SKIP includes 1) — D1 still holds original D1 value
X_train['uid'] = (X_train['card1_addr1'].astype(str) + '_' +
                  np.floor(X_train['day'] - X_train['D1']).astype(str))
X_test['uid']  = (X_test['card1_addr1'].astype(str) + '_' +
                  np.floor(X_test['day']  - X_test['D1']).astype(str))

# Save relation columns for graph construction BEFORE any further encoding
uid_train    = X_train['uid'].copy()
card1_train  = X_train['card1'].copy()
pemail_train = X_train['P_emaildomain'].copy()
device_train = X_train['DeviceInfo'].copy()

print('UID and relation columns saved for graph construction.')
print(f'Unique UIDs: {uid_train.nunique():,}')


In [ ]:
# ── Frequency, aggregation, count-unique encodings (same as experiment_tcn) 
print('Frequency encoding:')
encode_FE(X_train, X_test, ['addr1', 'card1', 'card2', 'card3', 'P_emaildomain'])
encode_FE(X_train, X_test, ['card1_addr1', 'card1_addr1_P_emaildomain'])
encode_FE(X_train, X_test, ['uid'])

print('\nAggregation features:')
encode_AG(
    ['TransactionAmt', 'D9', 'D11'],
    ['card1', 'card1_addr1', 'card1_addr1_P_emaildomain'],
    ['mean', 'std'], usena=True
)
encode_AG(
    ['TransactionAmt', 'D4', 'D9', 'D10', 'D15'],
    ['uid'], ['mean', 'std'], fillna=True, usena=True
)
encode_AG(
    ['C' + str(x) for x in range(1, 15) if x != 3],
    ['uid'], ['mean'], fillna=True, usena=True
)
encode_AG(
    ['M' + str(x) for x in range(1, 10)],
    ['uid'], ['mean'], fillna=True, usena=True
)
encode_AG(['C14'], ['uid'], ['std'], fillna=True, usena=True)

print('\nCount-unique features:')
encode_AG2(['P_emaildomain', 'dist1', 'DT_M', 'id_02', 'cents'], ['uid'])
encode_AG2(['C13', 'V314'], ['uid'])
encode_AG2(['V127', 'V136', 'V309', 'V307', 'V320'], ['uid'])

X_train['outsider15'] = (np.abs(X_train['D1'] - X_train['D15']) > 3).astype('int8')
X_test['outsider15']  = (np.abs(X_test['D1']  - X_test['D15']) > 3).astype('int8')

print('\nDone.')


In [ ]:
# ── Final column selection (same drop list as experiment_tcn) ──────────────
DROP_COLS = (
    ['TransactionDT', 'TransactionID'] +
    ['D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14'] +
    ['DT_M', 'day', 'uid'] +
    ['C3', 'M5', 'id_08', 'id_33'] +
    ['card4', 'id_07', 'id_14', 'id_21', 'id_30', 'id_32', 'id_34'] +
    ['id_' + str(x) for x in range(22, 28)]
)
cols = [c for c in X_train.columns if c in X_train.columns and c not in DROP_COLS]

X_catgnn = X_train[cols].copy().fillna(-1).astype('float32').values
y_catgnn = y_train.values.astype('int64')

IN_DIM = X_catgnn.shape[1]
print(f'Node feature matrix: {X_catgnn.shape}  ->  IN_DIM = {IN_DIM}')
print(f'Labels: {y_catgnn.shape}, fraud rate = {y_catgnn.mean()*100:.2f}%')


In [ ]:
# ── Stratified split on node IDs (transductive: all nodes in one graph) ───
all_nids = np.arange(len(y_catgnn))
train_nids, val_nids = train_test_split(
    all_nids, test_size=0.2, stratify=y_catgnn, random_state=42
)
train_nids = torch.LongTensor(train_nids)
val_nids   = torch.LongTensor(val_nids)

print(f'Train nodes: {len(train_nids):,} | Val nodes: {len(val_nids):,}')
print(f'Train fraud: {y_catgnn[train_nids].mean()*100:.2f}% | '
      f'Val fraud: {y_catgnn[val_nids].mean()*100:.2f}%')

# StandardScaler fitted on train nodes only (same as experiment_tcn)
scaler = StandardScaler()
X_catgnn[train_nids.numpy()] = scaler.fit_transform(X_catgnn[train_nids.numpy()])
X_catgnn[val_nids.numpy()]   = scaler.transform(X_catgnn[val_nids.numpy()])

node_features = torch.FloatTensor(X_catgnn)
node_labels   = torch.LongTensor(y_catgnn)

print(f'Feature tensor: {node_features.shape}, Labels tensor: {node_labels.shape}')


---
## 2. Graph Construction (GTAN Methodology)

**Identical to `experiment_gat.ipynb`** — GTAN temporal graph with K=3 past neighbors per relation (uid, card1, P_emaildomain, DeviceInfo), directed forward-in-time edges, reverse edges + self-loops added. Hub groups (>5000 nodes) skipped.

Reference: Xiang et al., *Semi-supervised Credit Card Fraud Detection via Attribute-driven Graph Representation* (AAAI 2023)


In [ ]:
def build_gtan_graph(n_nodes, relation_series_dict, k=3, max_group=5000):
    """
    Build GTAN-style homogeneous transaction graph.
    For each relation: group by attribute, connect each transaction to
    its k most recent preceding transactions in the same group.
    Adds reverse edges + self-loops for bidirectional message passing.
    """
    src_list, dst_list = [], []
    stats = {}

    for rel_name, series in relation_series_dict.items():
        n_before = len(src_list)
        skipped_large = skipped_nan = 0

        for group_val, group_idx in series.groupby(series).groups.items():
            if pd.isna(group_val) or group_val in (-1, '-1', 'nan'):
                skipped_nan += 1
                continue
            indices = list(group_idx)
            n = len(indices)
            if n < 2:
                continue
            if n > max_group:
                skipped_large += 1
                continue
            for i in range(1, n):
                dst = indices[i]
                for src in indices[max(0, i-k):i]:
                    src_list.append(src)
                    dst_list.append(dst)

        n_added = len(src_list) - n_before
        stats[rel_name] = {'edges': n_added,
                           'skipped_large': skipped_large,
                           'skipped_nan':   skipped_nan}
        print(f'  [{rel_name}] edges={n_added:,}  '
              f'skipped_large={skipped_large}  skipped_nan={skipped_nan}')

    if len(src_list) == 0:
        print('  WARNING: No edges built.')
        g = dgl.graph(([], []), num_nodes=n_nodes)
    else:
        src_arr = np.array(src_list, dtype=np.int64)
        dst_arr = np.array(dst_list, dtype=np.int64)
        g = dgl.graph((src_arr, dst_arr), num_nodes=n_nodes)

    g = dgl.add_reverse_edges(g)
    g = dgl.add_self_loop(g)

    print(f'\nFinal graph: {g.num_nodes():,} nodes, {g.num_edges():,} edges')
    print(f'Avg degree: {g.num_edges() / g.num_nodes():.1f}')
    return g, stats

print('Graph builder defined.')


In [ ]:
# ── Build GTAN graph on all training transactions ─────────────────────────
relation_series = {
    'uid':           uid_train.reset_index(drop=True),
    'card1':         card1_train.reset_index(drop=True),
    'P_emaildomain': pemail_train.reset_index(drop=True),
    'DeviceInfo':    device_train.reset_index(drop=True),
}

print(f'Building GTAN graph on {len(y_catgnn):,} transactions...')
print(f'K={GRAPH_K_NEIGHBORS} temporal neighbors, max_group={MAX_GROUP_SIZE}')
print()

t0 = time.time()
g, graph_stats = build_gtan_graph(
    n_nodes=len(y_catgnn),
    relation_series_dict=relation_series,
    k=GRAPH_K_NEIGHBORS,
    max_group=MAX_GROUP_SIZE
)
print(f'\nGraph built in {time.time()-t0:.1f}s')

g.ndata['feat']  = node_features
g.ndata['label'] = node_labels

with open(os.path.join(SAVE_DIR, 'graph_stats.json'), 'w') as f:
    json.dump({
        'n_nodes': g.num_nodes(),
        'n_edges': g.num_edges(),
        'avg_degree': g.num_edges() / g.num_nodes(),
        'relation_stats': graph_stats
    }, f, indent=2)


In [ ]:
# ── Graph statistics ──────────────────────────────────────────────────────
in_deg   = g.in_degrees().numpy()
out_deg  = g.out_degrees().numpy()
total_deg = in_deg + out_deg
fraud_mask  = y_catgnn == 1
benign_mask = y_catgnn == 0

print('=== Graph Statistics ===')
print(f'Nodes:            {g.num_nodes():>10,}')
print(f'Edges:            {g.num_edges():>10,}')
print(f'Avg total degree: {total_deg.mean():>10.2f}')
print(f'Max total degree: {total_deg.max():>10,}')
print()
print(f'Fraud node avg degree:  {total_deg[fraud_mask].mean():.2f}')
print(f'Benign node avg degree: {total_deg[benign_mask].mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(np.clip(total_deg, 0, 50), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Degree Distribution (clipped at 50)')
axes[0].set_xlabel('Degree'); axes[0].set_ylabel('Count')
axes[1].hist(np.clip(total_deg[fraud_mask],  0, 50), bins=30, alpha=0.7, label='Fraud',  color='red')
axes[1].hist(np.clip(total_deg[benign_mask], 0, 50), bins=30, alpha=0.5, label='Benign', color='blue')
axes[1].set_title('Degree Distribution by Label')
axes[1].set_xlabel('Degree'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'degree_distribution.png'), dpi=100)
plt.show()


---
## 3. CaT-GNN Model Architecture

**Based on:** Duan et al., arXiv:2402.14708 (Nov 2024) + Architecture Justification v2 §4.5

**Three components:**

```
CausalInspector (1 layer):
  Input: DGLBlock + source features (n_src, in_dim)
  Step 1: GATConv(in_dim→64, 4 heads, get_attention=True)  ← scoring only
  Step 2: Mean attention across heads → importance score per edge (n_edges,)
  Step 3: Per-dst mean threshold → causal mask / env mask
  Step 4: Aggregate ORIGINAL src features with masks  ← Bug-1 fix
  Step 5: Linear(in_dim→64) for each set → h_causal, h_env  (n_dst, 64)

CausalIntervener (PL variant, training only):
  blend_net: Linear(128→64) → ReLU → Linear(64→2) → Softmax  ← Bug-2 fix (not Beta)
  h_env_intervened = w[:,0]*h_causal + w[:,1]*h_env
  [At inference: h_env_intervened = h_env, no intervention]

Combination:
  h_combined = h_causal + h_env_intervened
  LayerNorm(64) → Linear(64→64) → ReLU → Dropout(0.3)
  → (n_dst, 64) causal embedding

FraudClassifier: Linear(64→32) → ReLU → Dropout(0.2) → Linear(32→1)
```

**3 bugs fixed (arch_justification_v2 §2.5):**
1. Aggregate original src features, not post-GAT features
2. Learned blend weights (PL) not random Beta noise (RR)
3. DGLBlock srcdata/dstdata handling for bipartite blocks


In [ ]:
# ── ROOT CAUSE ANALYSIS & FIXES ─────────────────────────────────────────
# RC-1 (CRITICAL): GATConv computed attention on raw 263-d features → noisy attention
#        FIX: Feature pre-embedding MLP (263→128) before GATConv
#             Better feature space = meaningful attention = better causal/env split
# RC-2 (CRITICAL): causal_proj/env_proj were flat Linear (no ReLU) → linear-only
#        FIX: Add ReLU after each projection
# RC-3 (SIGNIFICANT): Hard binary mask discarded attention magnitudes in aggregation
#        FIX: Soft attention-weighted aggregation — weight each neighbor by
#             its attention value (causal) or (threshold-attention) (env)
# RC-4 (SIGNIFICANT): h_causal + h_env_intervened element-wise sum collapsed
#        FIX: LayerNorm on outputs + cat([h_causal, h_env]) → Linear(128→64)
# RC-5 (MODERATE): LR=0.0005 too slow, FAN_OUT=[10] too sparse
#        FIX: LR=0.001, FAN_OUT=[15] (see config cell)

class CausalInspector(nn.Module):
    """
    Improved CausalInspector with 5-fix patches applied.

    Architecture:
      feat_embed: Linear(in_dim→128) → LayerNorm(128) → ReLU → Dropout  [RC-1 fix]
      GATConv(128→64, 4 heads) for attention scoring on embedded features
      soft attention-weighted aggregation (not hard binary mean mask)      [RC-3 fix]
      causal_proj: Linear(128→64) → ReLU                                  [RC-2 fix]
      env_proj:    Linear(128→64) → ReLU
    """
    def __init__(self, in_dim, hidden_dim=64, num_heads=4,
                 feat_drop=0.3, attn_drop=0.3):
        super().__init__()
        embed_dim = hidden_dim * 2  # 128

        # RC-1 FIX: Pre-embed raw features before attention computation
        # Raw 263-d tabular features → compact 128-d embedding
        # Makes GATConv attention scores reflect actual feature similarity
        self.feat_embed = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(feat_drop)
        )

        # GATConv on embedded 128-d features (better attention quality)
        self.attn_layer = GATConv(
            in_feats    = embed_dim,
            out_feats   = hidden_dim,
            num_heads   = num_heads,
            feat_drop   = feat_drop,
            attn_drop   = attn_drop,
            activation  = F.elu,
            residual    = False,
            allow_zero_in_degree = True
        )

        # RC-2 FIX: ReLU in projections (was flat Linear, no nonlinearity)
        # Projects aggregated 128-d embedded features → 64-d with nonlinearity
        self.causal_proj = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU())
        self.env_proj    = nn.Sequential(nn.Linear(embed_dim, hidden_dim), nn.ReLU())

    def forward(self, g, h):
        """
        g: DGLBlock (mini-batch) or DGLGraph (full-graph)
        h: source node features (n_src, in_dim)
        Returns: h_causal (n_dst, 64), h_env (n_dst, 64)
        """
        # ── Step 0: RC-1 FIX — pre-embed features ─────────────────────────
        h_embed = self.feat_embed(h)  # (n_src, 128)

        # ── Step 1: GATConv on embedded features for attention scores ──────
        if g.is_block:
            n_dst       = g.num_dst_nodes()
            h_dst_embed = h_embed[:n_dst]  # dst nodes = first n_dst rows
            _, attn     = self.attn_layer(g, (h_embed, h_dst_embed), get_attention=True)
            h_src       = h_embed
        else:
            n_dst   = g.num_nodes()
            _, attn = self.attn_layer(g, h_embed, get_attention=True)
            h_src   = h_embed

        # ── Step 2: Per-edge importance (mean across heads) ────────────────
        edge_importance = attn.mean(dim=1).squeeze(-1)  # (n_edges,)

        # ── Step 3: Per-dst mean threshold ─────────────────────────────────
        src_idx, dst_idx = g.edges()
        dst_sum   = torch.zeros(n_dst, device=edge_importance.device)
        dst_count = torch.zeros(n_dst, device=edge_importance.device)
        dst_sum.scatter_add_(0, dst_idx, edge_importance)
        dst_count.scatter_add_(0, dst_idx, torch.ones_like(edge_importance))
        dst_mean  = dst_sum / (dst_count + 1e-8)
        threshold = dst_mean[dst_idx]

        # ── Step 4: RC-3 FIX — soft attention-weighted aggregation ─────────
        # Hard binary mask discarded attention magnitude; soft weighting retains it:
        #   Causal set: weighted by attention value (higher = more trusted = more weight)
        #   Env set:    weighted by (threshold - attention) (lower = more spurious = more env)
        causal_mask = (edge_importance >= threshold).float()
        env_mask    = 1.0 - causal_mask

        causal_w = edge_importance * causal_mask                              # soft causal weight
        env_w    = (threshold - edge_importance).clamp(min=0) * env_mask      # soft env weight

        embed_dim = h_src.shape[-1]  # 128
        dst_exp   = dst_idx.unsqueeze(-1).expand(-1, embed_dim)

        causal_feat = h_src[src_idx] * causal_w.unsqueeze(-1)
        env_feat    = h_src[src_idx] * env_w.unsqueeze(-1)

        h_causal_sum = torch.zeros(n_dst, embed_dim, device=h_src.device)
        h_env_sum    = torch.zeros(n_dst, embed_dim, device=h_src.device)
        causal_wsum  = torch.zeros(n_dst, device=h_src.device)
        env_wsum     = torch.zeros(n_dst, device=h_src.device)

        h_causal_sum.scatter_add_(0, dst_exp, causal_feat)
        h_env_sum.scatter_add_(0, dst_exp, env_feat)
        causal_wsum.scatter_add_(0, dst_idx, causal_w)
        env_wsum.scatter_add_(0, dst_idx, env_w)

        h_causal_agg = h_causal_sum / (causal_wsum.unsqueeze(-1) + 1e-8)
        h_env_agg    = h_env_sum    / (env_wsum.unsqueeze(-1)    + 1e-8)

        # Fallback: all-neighbor mean when one set is empty
        no_causal = (causal_wsum == 0).unsqueeze(-1)
        no_env    = (env_wsum    == 0).unsqueeze(-1)
        if no_causal.any() or no_env.any():
            h_all   = torch.zeros(n_dst, embed_dim, device=h_src.device)
            all_cnt = torch.zeros(n_dst, device=h_src.device)
            h_all.scatter_add_(0, dst_exp, h_src[src_idx])
            all_cnt.scatter_add_(0, dst_idx, torch.ones_like(causal_w))
            h_all = h_all / (all_cnt.unsqueeze(-1) + 1e-8)
            h_causal_agg = torch.where(no_causal, h_all, h_causal_agg)
            h_env_agg    = torch.where(no_env,    h_all, h_env_agg)

        # ── Step 5: RC-2 FIX — project with ReLU ──────────────────────────
        h_causal = self.causal_proj(h_causal_agg)  # (n_dst, 64)
        h_env    = self.env_proj(h_env_agg)          # (n_dst, 64)
        return h_causal, h_env


class CausalIntervener(nn.Module):
    """
    Causal Intervener — PL (Proportional + Learned) variant. Unchanged.
    Training: h_env_intervened = w0*h_causal + w1*h_env  (learned softmax blend)
    Inference: h_env_intervened = h_env  (no intervention)
    """
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.blend_net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, h_causal, h_env):
        if self.training:
            weights = F.softmax(
                self.blend_net(torch.cat([h_causal, h_env], dim=-1)), dim=-1)
            return weights[:, 0:1] * h_causal + weights[:, 1:2] * h_env
        return h_env


class CausalTemporalGNN(nn.Module):
    """
    Full CaT-GNN with RC-4 fix: LayerNorm on inspector outputs
    + concatenation-based combination instead of element-wise sum.

    RC-4 FIX details:
      Before: h_combined = h_causal + h_env_intervened  (sum of two linear projections)
      After:  LayerNorm(h_causal), LayerNorm(h_env)
              → cat([h_causal, h_env_intervened]) → Linear(128→64) → ReLU
              This gives the model freedom to weight causal vs env differently.
    """
    def __init__(self, in_dim, hidden_dim=64, num_heads=4,
                 feat_drop=0.3, attn_drop=0.3):
        super().__init__()
        self.inspector  = CausalInspector(in_dim, hidden_dim, num_heads,
                                          feat_drop, attn_drop)
        self.intervener = CausalIntervener(hidden_dim)

        # RC-4 FIX: normalize before intervener; concat instead of sum
        self.ln_causal = nn.LayerNorm(hidden_dim)
        self.ln_env    = nn.LayerNorm(hidden_dim)
        self.combine   = nn.Linear(hidden_dim * 2, hidden_dim)  # 128 → 64

        self.norm        = nn.LayerNorm(hidden_dim)
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(feat_drop)
        )

    def forward(self, g, x):
        h_causal, h_env = self.inspector(g, x)

        # RC-4 FIX: normalize before intervener (prevents scale collapse)
        h_causal = self.ln_causal(h_causal)
        h_env    = self.ln_env(h_env)

        h_env_intervened = self.intervener(h_causal, h_env)

        # RC-4 FIX: concatenate → project (more expressive than sum)
        h_combined = F.relu(
            self.combine(torch.cat([h_causal, h_env_intervened], dim=-1))
        )
        h_combined = self.norm(h_combined)
        return self.output_proj(h_combined)   # (n_dst, 64)


class FraudClassifier(nn.Module):
    """MLP head: 64 → 32 → 1. Same as arch_justification_v2 §4.7."""
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, h): return self.fc(h)


# ── Loss: weighted BCE from experiment_tcn ────────────────────────────────
class_weights_arr = compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1]), y=y_catgnn
)
pos_weight_val = torch.tensor(
    [class_weights_arr[1] / class_weights_arr[0]], dtype=torch.float32
)

def weighted_bce_loss(logits, targets):
    pw = pos_weight_val.to(logits.device)
    return F.binary_cross_entropy_with_logits(logits, targets.float(), pos_weight=pw)


# ── Instantiate ───────────────────────────────────────────────────────────
catgnn_model = CausalTemporalGNN(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS, feat_drop=FEAT_DROP, attn_drop=ATTN_DROP
).to(DEVICE)
classifier = FraudClassifier(hidden_dim=HIDDEN_DIM).to(DEVICE)

n_params_model = sum(p.numel() for p in catgnn_model.parameters())
n_params_cls   = sum(p.numel() for p in classifier.parameters())
print(f'CaT-GNN parameters:    {n_params_model:,}')
print(f'Classifier parameters: {n_params_cls:,}')
print(f'Total parameters:      {n_params_model + n_params_cls:,}')
print(f'pos_weight (fraud):    {pos_weight_val.item():.2f}')
print()
print('Fixes applied:')
print('  RC-1: feat_embed Linear(263→128)+LayerNorm+ReLU before GATConv')
print('  RC-2: ReLU added to causal_proj and env_proj')
print('  RC-3: Soft attention-weighted aggregation (not hard binary mask)')
print('  RC-4: LayerNorm on h_causal/h_env + cat→Linear(128→64) combination')
print('  RC-5: LR=0.001, FAN_OUT=[15] (in config cell)')


---
## 4. Training Pipeline

Mini-batch: `NeighborSampler([15])` — 1-layer CaT-GNN samples **15 neighbors** (raised from 10 for more stable causal/env split).

Validation: `MultiLayerFullNeighborSampler(1)` — full 1-hop neighborhood.

**Key training behavior:**
- `CausalIntervener` active only during `model.train()` — breaks spurious correlations
- At `model.eval()`: `h_env_intervened = h_env` (clean environment signal)
- `LR=0.001` (raised from 0.0005 — paper uses 0.003; 0.0005 caused underfitting)


In [ ]:
# ── Optimizer & Scheduler — from experiment_tcn ──────────────────────────
optimizer = torch.optim.Adam(
    list(catgnn_model.parameters()) + list(classifier.parameters()),
    lr=LR,            # 0.0005
    weight_decay=1e-5
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=LR_FACTOR,    # 0.5
    patience=PATIENCE_LR, # 10
    min_lr=MIN_LR,       # 1e-7
    verbose=True
)

# ── Dataloaders ───────────────────────────────────────────────────────────
# CaT-GNN is 1-layer: FAN_OUT=[10]
train_sampler = dgl.dataloading.NeighborSampler(FAN_OUT)
train_loader  = dgl.dataloading.DataLoader(
    g, train_nids, train_sampler,
    batch_size=BATCH_SIZE,  # 512
    shuffle=True,
    drop_last=False,
    num_workers=0,
    device=DEVICE
)

# Validation: full neighborhood (1 layer)
val_sampler = dgl.dataloading.MultiLayerFullNeighborSampler(1)
val_loader  = dgl.dataloading.DataLoader(
    g, val_nids, val_sampler,
    batch_size=2048,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    device=DEVICE
)

print(f'Optimizer: Adam  lr={LR}  weight_decay=1e-5')
print(f'Scheduler: ReduceLROnPlateau  patience={PATIENCE_LR}  factor={LR_FACTOR}  min_lr={MIN_LR}')
print(f'Fan-out: {FAN_OUT}  (1-layer CaT-GNN)')
print(f'Train batches: {len(train_loader):,}  |  Val batches: {len(val_loader):,}')


In [ ]:
def train_epoch(loader):
    catgnn_model.train()   # activates CausalIntervener PL blend
    classifier.train()
    total_loss, n_batches = 0.0, 0

    for input_nodes, output_nodes, blocks in loader:
        # blocks[0]: single DGLBlock (1-layer CaT-GNN)
        x = blocks[0].srcdata['feat']  # (n_src, in_dim)
        y = node_labels[output_nodes.cpu()].to(DEVICE)

        # Forward through CaT-GNN + classifier
        h      = catgnn_model(blocks[0], x)      # (batch, hidden_dim)
        logits = classifier(h).squeeze(-1)        # (batch,)
        loss   = weighted_bce_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(catgnn_model.parameters()) + list(classifier.parameters()), 1.0
        )
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


@torch.no_grad()
def evaluate(loader):
    catgnn_model.eval()    # deactivates CausalIntervener (h_env_intervened = h_env)
    classifier.eval()
    all_probs, all_labels = [], []

    for input_nodes, output_nodes, blocks in loader:
        x      = blocks[0].srcdata['feat']
        h      = catgnn_model(blocks[0], x)
        logits = classifier(h).squeeze(-1)
        probs  = torch.sigmoid(logits).cpu().numpy()
        labels = node_labels[output_nodes.cpu()].numpy()
        all_probs.extend(probs)
        all_labels.extend(labels)

    probs  = np.array(all_probs)
    labels = np.array(all_labels)

    auc       = roc_auc_score(labels, probs)
    y_pred    = (probs >= 0.5).astype(int)
    f1        = f1_score(labels, y_pred, zero_division=0)
    precision = precision_score(labels, y_pred, zero_division=0)
    recall    = recall_score(labels, y_pred, zero_division=0)

    return {
        'auc': float(auc), 'f1': float(f1),
        'precision': float(precision), 'recall': float(recall)
    }, probs, labels

print('Train / evaluate functions defined.')
print('CausalIntervener: ACTIVE in train_epoch  |  INACTIVE in evaluate (eval mode)')


In [ ]:
# ── Training loop ────────────────────────────────────────────────────────
# EarlyStopping: monitor val_auc, patience=50 (experiment_tcn)
# ModelCheckpoint: save best val_auc
history = {
    'train_loss': [], 'val_auc': [], 'val_f1': [],
    'val_precision': [], 'val_recall': []
}
best_auc   = 0.0
no_improve = 0
best_probs = best_labels = None

print(f'Training for up to {EPOCHS} epochs  |  '
      f'EarlyStopping patience={PATIENCE_ES}  |  monitor=val_auc')
print(f'CausalIntervener: PL variant active during training')
print(f'{"Epoch":>6} {"TrainLoss":>10} {"ValAUC":>8} {"ValF1":>8} '
      f'{"ValPrec":>8} {"ValRecall":>9}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss               = train_epoch(train_loader)
    val_metrics, val_p, val_l = evaluate(val_loader)

    scheduler.step(val_metrics['auc'])

    history['train_loss'].append(train_loss)
    history['val_auc'].append(val_metrics['auc'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])

    if val_metrics['auc'] > best_auc:
        best_auc    = val_metrics['auc']
        no_improve  = 0
        best_probs  = val_p.copy()
        best_labels = val_l.copy()
        torch.save(catgnn_model.state_dict(),
                   os.path.join(SAVE_DIR, 'models', 'catgnn_best.pt'))
        torch.save(classifier.state_dict(),
                   os.path.join(SAVE_DIR, 'models', 'classifier_best.pt'))
        marker = ' *'
    else:
        no_improve += 1
        marker = ''

    if epoch % 10 == 0 or epoch == 1:
        elapsed = time.time() - t0
        print(f'{epoch:>6} {train_loss:>10.4f} '
              f'{val_metrics["auc"]:>8.4f} {val_metrics["f1"]:>8.4f} '
              f'{val_metrics["precision"]:>8.4f} {val_metrics["recall"]:>9.4f}  '
              f'({elapsed:.1f}s){marker}')

    if no_improve >= PATIENCE_ES:
        print(f'\nEarly stopping at epoch {epoch} (patience={PATIENCE_ES})')
        break

print(f'\nBest Val AUC: {best_auc:.4f}  '
      f'(epoch {int(np.argmax(history["val_auc"]))+1})')

with open(os.path.join(SAVE_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)


---
## 5. Evaluation & Analysis

- Optimal threshold search on validation set (maximizes F1)
- Full classification report at optimal threshold
- Causal attention visualization: which neighbor fraction is causal vs environment


In [ ]:
# ── Reload best model and evaluate ───────────────────────────────────────
catgnn_model.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, 'models', 'catgnn_best.pt'),
               map_location=DEVICE))
classifier.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, 'models', 'classifier_best.pt'),
               map_location=DEVICE))

_, best_probs, best_labels = evaluate(val_loader)

# ── Optimal threshold search (maximize F1) ─────────────────────────────
precisions, recalls, thresholds = precision_recall_curve(best_labels, best_probs)
f1_scores_thr = 2 * precisions[:-1] * recalls[:-1] / (
    precisions[:-1] + recalls[:-1] + 1e-8)
best_idx    = np.argmax(f1_scores_thr)
best_thresh = thresholds[best_idx]
best_f1_thr = f1_scores_thr[best_idx]

print(f'Default threshold (0.5):  '
      f'F1 = {f1_score(best_labels, (best_probs>=0.5).astype(int)):.4f}')
print(f'Optimal threshold ({best_thresh:.3f}): F1 = {best_f1_thr:.4f}')

# PR curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recalls[:-1], precisions[:-1], color='darkorange', lw=2)
axes[0].axvline(x=recalls[best_idx], color='red', linestyle='--',
                label=f'Best thresh={best_thresh:.3f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve'); axes[0].legend()

fpr, tpr, _ = roc_curve(best_labels, best_probs)
auc_score = roc_auc_score(best_labels, best_probs)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc_score:.4f}')
axes[1].plot([0,1],[0,1], 'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pr_roc_curves.png'), dpi=100)
plt.show()


In [ ]:
# ── Final metrics at optimal threshold ───────────────────────────────────
y_pred_opt = (best_probs >= best_thresh).astype(int)

print('=== CaT-GNN Final Validation Metrics ===')
print(f'AUC:              {roc_auc_score(best_labels, best_probs):.4f}')
print(f'F1  (thr={best_thresh:.3f}): {f1_score(best_labels, y_pred_opt):.4f}')
print(f'Precision:        {precision_score(best_labels, y_pred_opt):.4f}')
print(f'Recall:           {recall_score(best_labels, y_pred_opt):.4f}')
print()
print(classification_report(best_labels, y_pred_opt,
                             target_names=['Benign', 'Fraud']))

# Confusion matrix
cm = confusion_matrix(best_labels, y_pred_opt)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Benign', 'Pred Fraud'],
            yticklabels=['Actual Benign', 'Actual Fraud'], ax=ax)
ax.set_title(f'Confusion Matrix (threshold={best_thresh:.3f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=100)
plt.show()

# Save results
results = {
    'auc':        float(roc_auc_score(best_labels, best_probs)),
    'f1_optimal': float(f1_score(best_labels, y_pred_opt)),
    'precision':  float(precision_score(best_labels, y_pred_opt)),
    'recall':     float(recall_score(best_labels, y_pred_opt)),
    'threshold':  float(best_thresh)
}
with open(os.path.join(SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to {SAVE_DIR}/final_results.json')


In [ ]:
# ── Training history plots ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history['val_auc'], label='Val AUC', color='darkorange')
best_ep = int(np.argmax(history['val_auc'])) + 1
axes[1].axvline(x=best_ep, color='red', linestyle='--',
                label=f'Best epoch={best_ep}')
axes[1].set_title('Validation AUC')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].legend()

axes[2].plot(epochs_range, history['val_f1'],        label='F1',        color='green')
axes[2].plot(epochs_range, history['val_precision'], label='Precision',  color='blue')
axes[2].plot(epochs_range, history['val_recall'],    label='Recall',     color='orange')
axes[2].set_title('Validation F1 / Precision / Recall')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=100)
plt.show()


In [ ]:
# ── Causal attention analysis: fraction of neighbors classified as causal ─
# Reflects the improved soft-weighted inspector logic
@torch.no_grad()
def analyze_causal_split(n_batches=10):
    catgnn_model.eval()
    all_causal_fracs   = []
    fraud_causal_fracs = []
    benign_causal_fracs = []

    for i, (input_nodes, output_nodes, blocks) in enumerate(val_loader):
        if i >= n_batches:
            break
        g_block = blocks[0]
        x = g_block.srcdata['feat']
        labels_batch = node_labels[output_nodes.cpu()].numpy()
        inspector = catgnn_model.inspector
        n_dst = g_block.num_dst_nodes()

        # Step 0: pre-embed (RC-1 fix)
        h_embed = inspector.feat_embed(x)
        h_dst_embed = h_embed[:n_dst]
        _, attn = inspector.attn_layer(g_block, (h_embed, h_dst_embed),
                                        get_attention=True)
        edge_importance = attn.mean(dim=1).squeeze(-1)

        src_idx, dst_idx = g_block.edges()
        dst_sum   = torch.zeros(n_dst, device=edge_importance.device)
        dst_count = torch.zeros(n_dst, device=edge_importance.device)
        dst_sum.scatter_add_(0, dst_idx, edge_importance)
        dst_count.scatter_add_(0, dst_idx, torch.ones_like(edge_importance))
        dst_mean  = dst_sum / (dst_count + 1e-8)
        threshold = dst_mean[dst_idx]

        # Causal/env split (mirrors inspector logic)
        causal_mask = (edge_importance >= threshold).float()
        causal_cnt  = torch.zeros(n_dst, device=edge_importance.device)
        total_cnt   = torch.zeros(n_dst, device=edge_importance.device)
        causal_cnt.scatter_add_(0, dst_idx, causal_mask)
        total_cnt.scatter_add_(0, dst_idx, torch.ones_like(causal_mask))
        frac = (causal_cnt / (total_cnt + 1e-8)).cpu().numpy()

        all_causal_fracs.extend(frac.tolist())
        fraud_causal_fracs.extend(frac[labels_batch == 1].tolist())
        benign_causal_fracs.extend(frac[labels_batch == 0].tolist())

    return (np.array(all_causal_fracs),
            np.array(fraud_causal_fracs),
            np.array(benign_causal_fracs))

all_frac, fraud_frac, benign_frac = analyze_causal_split(n_batches=10)

print(f'Overall mean causal fraction: {all_frac.mean():.3f} (expected ~0.5)')
print(f'Fraud nodes:  mean causal fraction = {fraud_frac.mean():.3f}')
print(f'Benign nodes: mean causal fraction = {benign_frac.mean():.3f}')
print()
if fraud_frac.mean() > benign_frac.mean():
    print('Causal split working: fraud nodes have MORE causal neighbors')
else:
    print('WARNING: fraud nodes have fewer causal neighbors — split still noisy')

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(fraud_frac,  bins=20, alpha=0.6, label='Fraud',  color='red',  density=True)
ax.hist(benign_frac, bins=20, alpha=0.6, label='Benign', color='blue', density=True)
ax.set_xlabel('Causal neighbor fraction per node')
ax.set_ylabel('Density')
ax.set_title('CausalInspector: Causal Neighbor Fraction (Fraud vs Benign)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'causal_fraction_dist.png'), dpi=100)
plt.show()


---
## 6. Summary

### Architecture

| Component | Details |
|-----------|----------|
| **CausalInspector** | GATConv(in_dim→64, 4 heads) for attention scoring; scatter-aggregate original features; causal_proj + env_proj (in_dim→64 each) |
| **CausalIntervener** | PL: blend_net(128→64→2) + softmax; active ONLY during training |
| **Combination** | h_causal + h_env_intervened → LayerNorm → Linear(64→64) → ReLU → Dropout(0.3) |
| **FraudClassifier** | Linear(64→32) → ReLU → Dropout(0.2) → Linear(32→1) |

### Bugs Fixed (arch_justification_v2 §2.5)

1. **Source features** (not post-GAT): CausalInspector aggregates `g.srcdata['feat']`, scores with `attn_layer` but does NOT use its output embeddings
2. **Learned blend** (not Beta): CausalIntervener uses `blend_net` MLP → softmax weights, implementing the PL variant (best: AUC 0.8281, F1 0.7211 on S-FFSD)
3. **DGLBlock compatibility**: uses `g.srcdata` / `g.dstdata`, handles bipartite src/dst split correctly; `h_dst = h_src[:n_dst]` (DGL guarantee)

### Training

- Loss: weighted BCE + pos_weight ≈ 27.4 (from experiment_tcn balanced class weights)
- Optimizer: Adam(lr=0.0005) — from experiment_tcn
- ReduceLROnPlateau(patience=10, factor=0.5, min_lr=1e-7) — from experiment_tcn
- EarlyStopping(patience=50, monitor=val_auc) — from experiment_tcn
- Mini-batch: FAN_OUT=[10], BATCH_SIZE=512 — 1-layer CaT-GNN
- Validation: MultiLayerFullNeighborSampler(1), batch_size=2048

### Output

Produces 64-d causal embedding per transaction, compatible with Gated Attention Fusion alongside TCN (64-d temporal) and GAT (64-d relational) embeddings.
